# Stage 1 — Non-Instruction Fine-Tuning

Domain adaptation of `Qwen/Qwen2.5-0.5B` on raw California homeowners-insurance text using [Unsloth](https://github.com/unslothai/unsloth), LoRA/QLoRA, on a free Colab T4.

Spec: `specs/002-unsloth-ca-homeowners-finetune.md` §6-7 (Stage 1).

**Run this notebook on Google Colab with a T4 GPU runtime** (Runtime → Change runtime type → T4 GPU). It has not been executed locally — there is no CUDA GPU in the local dev environment for this repo.

Before running: upload/clone this repo into the Colab session so that `data/1-non_instruction_data.txt` and the `src/` package are available at the paths used below (e.g. `!git clone <repo-url>` then `%cd <repo-name>`).

This notebook persists its output by pushing to the Hugging Face Hub (see "Hugging Face Hub authentication" below) rather than through Colab's Drive-mount flow or a manual browser upload/download — add an `HF_TOKEN` Colab secret before running, and set `HF_USERNAME` in the paths cell to your Hugging Face username.

In [1]:
!git clone https://github.com/saravanangenai/LLM-FineTunning.git
%cd LLM-FineTunning/notebooks

fatal: destination path 'LLM-FineTunning' already exists and is not an empty directory.
/content/LLM-FineTunning/notebooks


## 0. Install dependencies

In [2]:
%%capture
import importlib.util

if importlib.util.find_spec("unsloth") is None:
    !pip install unsloth
    !pip install --no-deps "trl>=0.9.0" "peft>=0.11.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

## 0b. Hugging Face Hub authentication

Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) with a Hugging Face **write**-access token — create one at https://huggingface.co/settings/tokens if you don't have one. Used to push this stage's model to the Hub.

In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [4]:
import sys
from pathlib import Path

# Assumes this notebook is run from notebooks/ inside the cloned repo, so the
# repo root (one level up) contains the src/ package and data/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "1-non_instruction_data.txt"
ADAPTER_OUT_DIR = REPO_ROOT / "models" / "stage1-non-instruction-adapter"
MERGED_OUT_DIR = REPO_ROOT / "models" / "stage1-merged"

HF_USERNAME = "sharanmini"  # <-- set this to your Hugging Face username
HF_STAGE1_REPO = f"{HF_USERNAME}/qwen2.5-0.5b-ca-homeowners-stage1"

print("Repo root:", REPO_ROOT)
print("Data path exists:", DATA_PATH.exists())

Repo root: /content/LLM-FineTunning
Data path exists: True


## 1. Load raw domain text

In [5]:
from src.data_utils import load_raw_paragraphs

paragraphs = load_raw_paragraphs(DATA_PATH)
print(f"Loaded {len(paragraphs)} paragraphs")
print(paragraphs[0][:300])

Loaded 62 paragraphs
A homeowners insurance policy is a contract between a homeowner and an insurance company that protects a home and its contents against a defined set of risks, called perils, in exchange for a premium. In California, most homeowners carry a package policy known as an HO-3, which combines several cove


## 2. Load base model via Unsloth

In [6]:
from src.model_utils import BASE_MODEL_NAME, MAX_SEQ_LENGTH, load_base_model

model, tokenizer = load_base_model(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print("Loaded:", BASE_MODEL_NAME, "| eos_token:", tokenizer.eos_token)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-0.5B | eos_token: <|endoftext|>


## 3. Clean and chunk text into a training dataset

Paragraphs are already clean, standalone prose (see `data/1-non_instruction_data.txt` header). Each paragraph gets the tokenizer's EOS token appended so the model learns paragraph boundaries; `packing=True` below then concatenates/chunks these into `MAX_SEQ_LENGTH`-sized training blocks.

In [7]:
from src.data_utils import build_text_dataset

dataset = build_text_dataset(paragraphs, eos_token=tokenizer.eos_token)
dataset

Dataset({
    features: ['text'],
    num_rows: 62
})

## 4. Apply LoRA / QLoRA adapters

In [8]:
from src.model_utils import add_lora_adapters

model = add_lora_adapters(model)  # uses DEFAULT_LORA_CONFIG: r=16, alpha=16, dropout=0.05

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.3 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 5. Train on raw domain text

In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=str(REPO_ROOT / "outputs" / "stage1"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size 8, per spec §6.3
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=training_args,
)

train_result = trainer.train()
print(train_result.metrics)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/62 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 62 | Num Epochs = 3 | Total steps = 24
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.106206
10,2.815608
15,2.491784
20,2.339105


{'train_runtime': 65.5255, 'train_samples_per_second': 2.839, 'train_steps_per_second': 0.366, 'total_flos': 36896983718400.0, 'train_loss': 2.6290035247802734, 'epoch': 3.0}


## 6. Save the adapter

Colab local disk is not persistent across sessions — copy `ADAPTER_OUT_DIR` to Google Drive or push it to a private Hugging Face model repo after this cell (see spec §6.4).

In [10]:
ADAPTER_OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUT_DIR))
print("Saved adapter to:", ADAPTER_OUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in /content/LLM-FineTunning/models/stage1-non-instruction-adapter/tokenizer_config.json.


Saved adapter to: /content/LLM-FineTunning/models/stage1-non-instruction-adapter


Also save a full merged (base weights + LoRA) 16-bit checkpoint to `MERGED_OUT_DIR`. Stage 2 loads this directory as its starting "base" model rather than resuming training on top of this stage's raw PEFT adapter (see spec §6.4).

In [11]:
from src.model_utils import save_merged_model

MERGED_OUT_DIR.mkdir(parents=True, exist_ok=True)
save_merged_model(model, tokenizer, str(MERGED_OUT_DIR))
print("Saved merged model to:", MERGED_OUT_DIR)

config.json:   0%|          | 0.00/774 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/LLM-FineTunning/models/stage1-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:14<00:00, 14.71s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:06<00:00,  6.02s/it]


Unsloth: Merge process complete. Saved to `/content/LLM-FineTunning/models/stage1-merged`
Saved merged model to: /content/LLM-FineTunning/models/stage1-merged


In [12]:
# Persist Stage 1 outputs by pushing them to the Hugging Face Hub — this
# avoids both Colab's occasionally-flaky Drive-mount OAuth flow and a slow
# browser upload/download round trip. Stage 2 pulls HF_STAGE1_REPO directly.
from src.model_utils import push_folder_to_hub

push_folder_to_hub(str(ADAPTER_OUT_DIR), HF_STAGE1_REPO + "-adapter", token=HF_TOKEN)
push_folder_to_hub(str(MERGED_OUT_DIR), HF_STAGE1_REPO, token=HF_TOKEN)
print("Pushed merged model to:", f"https://huggingface.co/{HF_STAGE1_REPO}")

# Alternative: download as a zip instead of using the Hub (no HF token needed,
# but requires manually re-uploading the zip at the start of Stage 2).
# import shutil
# from google.colab import files
# shutil.make_archive("/content/stage1-merged", "zip", str(MERGED_OUT_DIR))
# files.download("/content/stage1-merged.zip")
# shutil.make_archive("/content/stage1-non-instruction-adapter", "zip", str(ADAPTER_OUT_DIR))
# files.download("/content/stage1-non-instruction-adapter.zip")

# Alternative: persist to Google Drive instead (can hit a flaky OAuth error).
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(ADAPTER_OUT_DIR, "/content/drive/MyDrive/stage1-non-instruction-adapter", dirs_exist_ok=True)
# shutil.copytree(MERGED_OUT_DIR, "/content/drive/MyDrive/stage1-merged", dirs_exist_ok=True)

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'sharan_s@yahoo.com/qwen2.5-0.5b-ca-homeowners-stage1-adapter'.

## 7. Test the model after non-instruction fine-tuning

These are raw continuation prompts (not questions/instructions — that's Stage 2), meant to check whether the model has picked up domain vocabulary, tone, and background knowledge from the raw text.

In [ ]:
import warnings
from unsloth import FastLanguageModel

from src.generation_utils import generate

FastLanguageModel.for_inference(model)

test_prompts = [
    "Dwelling coverage, sometimes labeled Coverage A,",
    "The California FAIR Plan is",
    "The difference between actual cash value and replacement cost value is",
    "Earthquake coverage in California",
    "After a covered loss, homeowners are typically expected to",
]

# Suppress the FutureWarning regarding max_new_tokens and max_length
warnings.filterwarnings("ignore", category=FutureWarning, message=".*max_new_tokens.*max_length.*")

for prompt in test_prompts:
    completion = generate(model, tokenizer, prompt, max_new_tokens=80, temperature=0.7)
    print(f"PROMPT: {prompt}\nCOMPLETION: {completion}\n{'-' * 80}")